# Model II

Biblioteki

In [1]:
from datetime import datetime
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score
)

import time
from model import MultiInstrumentCNN
from trening import get_device, train_model
from przygotowanie_danych import (INSTRUMENTS, create_datasets, create_loaders)

### Konfiguracja eksperymentu

In [10]:
dataset = "2_10"
model_name = "CNN2"

epochs = 100
patience = 10
batch_size = 8
learning_rate = 0.001

dropout2 = 0.1
dropout3 = 0.3

threshold = 0.8

results_file = Path("wyniki/modele.csv")

Sprawdzenie, czy eksperyment był już wykonywany

In [11]:
if results_file.exists():
    results = pd.read_csv(results_file)

    combination_exists = (
        (results["Dataset"] == dataset) &
        (results["Model"] == model_name) &
        (results["Batch size"] == batch_size) &
        (results["Learning rate"] == learning_rate) &
        (results["Dropout 2"] == dropout2) &
        (results["Dropout 3"] == dropout3)
    ).any()

    if combination_exists:
        raise ValueError(
            "Ta kombinacja datasetu i hiperparametrów "
            "była już testowana. Zatrzymanie programu."
        )

print("Nowa kombinacja. Kontynuacja wykonywania kodu.")


Nowa kombinacja. Kontynuacja wykonywania kodu.


### Nadanie identyfikatora eksperymentu

In [12]:
results_dir = Path("wyniki")
results_dir.mkdir(parents=True, exist_ok=True)

existing_numbers = [
    int(f.stem)
    for f in results_dir.glob("*.pth")
    if f.stem.isdigit()
]

learning_id = max(existing_numbers, default=0) + 1

weights_filename = f"{learning_id}.pth"

print(f"Learning ID: {learning_id}")
print(f"Plik wag: {weights_filename}")

Learning ID: 10
Plik wag: 10.pth


### Przygotowanie danych

In [13]:
train_dataset, val_dataset, test_dataset = create_datasets(dataset)

train_loader, val_loader, test_loader = create_loaders(
    train_dataset,
    val_dataset,
    test_dataset,
    batch_size=batch_size
)

print(f"Train: {len(train_dataset)}")
print(f"Validation: {len(val_dataset)}")
print(f"Test: {len(test_dataset)}")

Wczytuję spektrogramy z: D:\spectrograms\2_10
Train: 7758
Validation: 1658
Test: 1664


### Inicjalizacja modelu

In [14]:
device = get_device()
print(f"Urządzenie: {device}")

model = MultiInstrumentCNN(num_instruments=len(INSTRUMENTS), dropout2=dropout2, dropout3=dropout3).to(device)

criterion = nn.MultiLabelSoftMarginLoss()

optimizer = optim.Adam(model.parameters(), lr=learning_rate)


Urządzenie: cuda


### Trening sieci

In [15]:
start_time = time.perf_counter()

model, history, best_epoch, best_val_loss = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=epochs,
    patience=patience,
    model_name=weights_filename
)

training_time = time.perf_counter() - start_time

print(f"Czas treningu: {training_time / 60:.2f} min")

Epoka 1/100 | Train loss: 0.6104 | Val loss: 0.5476
Epoka 2/100 | Train loss: 0.5294 | Val loss: 0.5104
Epoka 3/100 | Train loss: 0.4979 | Val loss: 0.4652
Epoka 4/100 | Train loss: 0.4680 | Val loss: 0.4441
Epoka 5/100 | Train loss: 0.4427 | Val loss: 0.4180
Epoka 6/100 | Train loss: 0.4192 | Val loss: 0.3875
Epoka 7/100 | Train loss: 0.4008 | Val loss: 0.3749
Epoka 8/100 | Train loss: 0.3867 | Val loss: 0.3620
Epoka 9/100 | Train loss: 0.3717 | Val loss: 0.3598
Epoka 10/100 | Train loss: 0.3607 | Val loss: 0.3390
Epoka 11/100 | Train loss: 0.3480 | Val loss: 0.3373
Epoka 12/100 | Train loss: 0.3381 | Val loss: 0.3165
Epoka 13/100 | Train loss: 0.3258 | Val loss: 0.3034
Epoka 14/100 | Train loss: 0.3189 | Val loss: 0.3158
Epoka 15/100 | Train loss: 0.3111 | Val loss: 0.2968
Epoka 16/100 | Train loss: 0.3038 | Val loss: 0.3081
Epoka 17/100 | Train loss: 0.2985 | Val loss: 0.2940
Epoka 18/100 | Train loss: 0.2923 | Val loss: 0.3032
Epoka 19/100 | Train loss: 0.2841 | Val loss: 0.2962
Ep

### Zapis informacji o eksperymencie

In [16]:
experiment = pd.DataFrame([{
    "Learning_ID": learning_id,
    "Dataset": dataset,
    "Model": model_name,
    "Batch size": batch_size,
    "Learning rate": learning_rate,
    "Dropout 2": dropout2,
    "Dropout 3": dropout3,
    "Threshold": threshold,
    "Best epoch": best_epoch,
    "Best val loss": best_val_loss,
    "Training time [min]": training_time / 60
}])

experiment.to_csv(
    results_file,
    mode="a",
    header=not results_file.exists(),
    index=False
)

history_df = pd.DataFrame(history)

history_df.to_csv(
    results_dir / f"history_{learning_id}.csv",
    index_label="epoch"
)

print(f"Informacje o eksperymencie zapisano w: {results_file}")
print(f"Historia uczenia: history_{learning_id}.csv")


Informacje o eksperymencie zapisano w: wyniki\modele.csv
Historia uczenia: history_10.csv
